<a href="https://colab.research.google.com/github/Faiq-danZ/worksafe-ai/blob/main/training/train_model_tabular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Inisialisasi dan Memuat Data Final

In [13]:
import pandas as pd
import joblib
import os
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Menghubungkan ke Google Drive
drive.mount('/content/drive')

# Jalur file data siap latih
path_data = '/content/drive/MyDrive/Data/Data/Processed/data_siap_training.csv'
df = pd.read_csv(path_data)

print("Data siap latih berhasil dimuat")
print("Jumlah baris dan kolom:", df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data siap latih berhasil dimuat
Jumlah baris dan kolom: (1016, 1073)


## 2. Pemisahan Fitur dan Target

In [14]:
# Cek semua nama kolom yang ada di dataset
print(df.columns.tolist())

['occupation_code', 'title', 'description', 'skill_count', 'skill_importance_avg', 'skill_level_avg', 'top_skills', 'task_count', 'core_task_count', 'supplemental_task_count', 'incumbents_responding_avg', 'core_task_ratio', 'top_core_tasks', 'data_analysis_score', 'social_score', 'coaching_and_developing_others', 'communicating_with_people_outside_the_organization', 'communicating_with_supervisors,_peers,_or_subordinates', 'controlling_machines_and_processes', 'coordinating_the_work_and_activities_of_others', 'developing_objectives_and_strategies', 'developing_and_building_teams', 'documenting_recording_information', 'drafting,_laying_out,_and_specifying_technical_devices,_parts,_and_equipment', 'establishing_and_maintaining_interpersonal_relationships', 'estimating_the_quantifiable_characteristics_of_products,_events,_or_information', 'evaluating_information_to_determine_compliance_with_standards', 'getting_information', 'guiding,_directing,_and_motivating_subordinates', 'object_handl

In [15]:
# Daftar kolom yang akan dibuang dari fitur (X)
cols_to_drop = [
    'occupation_code', 'title', 'description', 'top_skills',
    'top_core_tasks', 'risk_label', 'reskilling_recommendation',
    'job_zone', 'job_zone_norm'
]

# Memisahkan Fitur (X)
X = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')

# Menentukan Target (y) menggunakan nama kolom yang benar
y = df['job_zone']

# Membagi data menjadi 80% latihan dan 20% pengujian
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Pemisahan data berhasil!")
print(f"Target yang digunakan: job_zone")
print(f"Jumlah fitur yang digunakan: {X.shape[1]} kolom")

Pemisahan data berhasil!
Target yang digunakan: job_zone
Jumlah fitur yang digunakan: 1064 kolom


## 3. Pelatihan Model (Random Forest)

In [18]:
# Model Random Forest
model = RandomForestClassifier(
    n_estimators=500,         # Menambah jumlah pohon agar lebih stabil
    max_depth=None,           # Memberikan kebebasan pohon untuk tumbuh
    min_samples_split=2,      # Detail dalam membagi data
    random_state=42,
    class_weight='balanced',  # jika jumlah Job Zone 1, 2, 3 dst tidak seimbang
    n_jobs=-1                 # Menggunakan semua core CPU agar lebih cepat
)

print("Sedang melatih model RF Upgrade...")
model.fit(X_train, y_train)

# Tes akurasi
y_pred = model.predict(X_test)
print(f"Akurasi RF Setelah Upgrade: {accuracy_score(y_test, y_pred) * 100:.2f}%")

Sedang melatih model RF Upgrade...
Akurasi RF Setelah Upgrade: 80.39%


## 4. Simpan Model

In [19]:
import joblib
import os

# 1. Tentukan folder tujuan di Drive/Repo
path_folder = '/content/drive/MyDrive/Data/Data/models/tabular_model/'
os.makedirs(path_folder, exist_ok=True)

# 2. Simpan model yang akurasinya 80% tadi
model_name = 'model_worksafe_v1.pkl'
joblib.dump(model, path_folder + model_name)

# 3. Simpan juga daftar fitur (X.columns)
joblib.dump(X.columns.tolist(), path_folder + 'feature_names.pkl')

print(f"Berhasil! Model disimpan sebagai: {model_name}")

Berhasil! Model disimpan sebagai: model_worksafe_v1.pkl
